# LeetCode 1720 — Decode XORed Array bằng Python

**Link đề:** https://leetcode.com/problems/decode-xored-array/
**Độ khó:** Easy · **Chủ đề:** Array, Bit Manipulation

```python
class Solution:
    def decode(self, encoded: List[int], first: int) -> List[int]:
        ...
```

## Mục tiêu học
1. Hiểu phép **XOR (`^`)** trên từng bit, đọc số ở dạng nhị phân bằng `bin()` / `format(x, "b")`.
2. Nắm 4 tính chất vàng: `a ^ 0 = a`, `a ^ a = 0`, giao hoán, kết hợp ⇒ **XOR là phép tự nghịch đảo**.
3. Từ `encoded[i] = arr[i] ^ arr[i+1]` suy ra `arr[i+1] = arr[i] ^ encoded[i]`.
4. Đi từ cách "dò thử giá trị" (rất chậm) → giải mã tuần tự `O(n)` → **XOR tiền tố** bằng `itertools.accumulate`.
5. Trace từng bit, chứng minh tính duy nhất của đáp án, luyện các biến thể XOR.

## 1. Đề bài và ví dụ

Có một mảng ẩn `arr` gồm `n` số nguyên **không âm**. Nó được mã hoá thành `encoded` dài `n - 1` với
`encoded[i] = arr[i] XOR arr[i + 1]`.

Cho `encoded` và `first = arr[0]`. Trả về mảng gốc `arr`. Đề bảo đảm đáp án tồn tại và **duy nhất**.

| `encoded` | `first` | Kết quả `arr` | Kiểm tra |
|---|:---:|---|---|
| `[1,2,3]` | 1 | `[1,0,2,1]` | `1^0=1, 0^2=2, 2^1=3` |
| `[6,2,7,3]` | 4 | `[4,2,0,7,4]` | `4^2=6, 2^0=2, 0^7=7, 7^4=3` |

**Constraints:** `2 <= n <= 10^4`; `len(encoded) == n - 1`; `0 <= encoded[i] <= 10^5`; `0 <= first <= 10^5`.

### 1.1. Kiểm tra chiều "mã hoá" trước

Muốn hiểu giải mã, hãy tự mã hoá: từ `arr` tính `encoded`. Trong Python XOR là toán tử `^` (**không phải** luỹ thừa! Luỹ thừa là `**`).

In [1]:
from typing import List

def encode(arr: List[int]) -> List[int]:
    return [arr[i] ^ arr[i + 1] for i in range(len(arr) - 1)]

print(encode([1, 0, 2, 1]))       # [1, 2, 3]
print(encode([4, 2, 0, 7, 4]))    # [6, 2, 7, 3]
print("2 ^ 3 =", 2 ^ 3, "| 2 ** 3 =", 2 ** 3)

[1, 2, 3]
[6, 2, 7, 3]
2 ^ 3 = 1 | 2 ** 3 = 8


## 2. Nền tảng Python và bit cần biết

### 2.1. Số nhị phân: `bin()` và `format(x, "04b")`

Mỗi số nguyên là dãy bit. `bin(6)` → `'0b110'`. `format(6, "04b")` → `'0110'` (đệm 0 cho đủ 4 chữ số) — tiện để in thẳng hàng.

In [2]:
for x in [0, 1, 2, 4, 6, 7]:
    print(f"{x} -> bin = {bin(x):>6} -> 4 bit = {format(x, '04b')}")

0 -> bin =    0b0 -> 4 bit = 0000
1 -> bin =    0b1 -> 4 bit = 0001
2 -> bin =   0b10 -> 4 bit = 0010
4 -> bin =  0b100 -> 4 bit = 0100
6 -> bin =  0b110 -> 4 bit = 0110
7 -> bin =  0b111 -> 4 bit = 0111


### 2.2. XOR trên từng bit

XOR so **từng cặp bit cùng vị trí**: hai bit **khác nhau** → `1`, **giống nhau** → `0`.

| a | b | a ^ b |
|:-:|:-:|:-:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

In [3]:
def show_xor(a, b_):
    print(f"  {format(a, '04b')}  ({a})")
    print(f"^ {format(b_, '04b')}  ({b_})")
    print(f"= {format(a ^ b_, '04b')}  ({a ^ b_})")

show_xor(4, 2)
print()
show_xor(7, 4)

  0100  (4)
^ 0010  (2)
= 0110  (6)

  0111  (7)
^ 0100  (4)
= 0011  (3)


### 2.3. Bốn tính chất vàng của XOR

1. `a ^ 0 == a` (XOR với 0 không đổi).
2. `a ^ a == 0` (tự triệt tiêu).
3. Giao hoán: `a ^ b == b ^ a`.
4. Kết hợp: `(a ^ b) ^ c == a ^ (b ^ c)`.

Hệ quả quan trọng: **`(a ^ b) ^ b == a`** — XOR với `b` hai lần thì trở về ban đầu. XOR vừa là phép "khoá" vừa là phép "mở khoá".

In [4]:
a, b_, c = 13, 6, 9
print("a ^ 0 =", a ^ 0)
print("a ^ a =", a ^ a)
print("a ^ b == b ^ a:", a ^ b_ == b_ ^ a)
print("(a ^ b) ^ c == a ^ (b ^ c):", (a ^ b_) ^ c == a ^ (b_ ^ c))
print("(a ^ b) ^ b =", (a ^ b_) ^ b_, "== a")

a ^ 0 = 13
a ^ a = 0
a ^ b == b ^ a: True
(a ^ b) ^ c == a ^ (b ^ c): True
(a ^ b) ^ b = 13 == a


### 2.4. Toán tử gán `^=` và `list.append`

`x ^= y` nghĩa là `x = x ^ y`. `arr.append(v)` thêm `v` vào cuối list.

In [5]:
x = 5
x ^= 3
print("5 ^ 3 =", x)
arr = [1]
arr.append(0); arr.append(2)
print(arr)

5 ^ 3 = 6
[1, 0, 2]


## 3. Cách trực giác — dò thử giá trị kế tiếp

Biết `arr[i]` và `encoded[i]`, ta cần `arr[i+1]` thoả `arr[i] ^ arr[i+1] == encoded[i]`. Người mới chưa biết tính chất XOR có thể **thử mọi giá trị** `v` từ 0 tới giới hạn. Vì `arr[i], encoded[i] ≤ 10^5 < 2^17`, đáp án luôn `< 2^17 = 131072`.

In [6]:
def decode_search(encoded: List[int], first: int, limit: int = 1 << 17) -> List[int]:
    arr = [first]
    for e in encoded:
        for v in range(limit):              # dò từng giá trị
            if arr[-1] ^ v == e:
                arr.append(v)
                break
    return arr

print("1 << 17 =", 1 << 17)
print(decode_search([1, 2, 3], 1))
print(decode_search([6, 2, 7, 3], 4))

1 << 17 = 131072
[1, 0, 2, 1]
[4, 2, 0, 7, 4]


### Chi phí và nút thắt

Mỗi vị trí có thể thử tới `2^17 ≈ 131.000` giá trị ⇒ worst `10^4 × 1.3·10^5 ≈ 1.3·10^9` phép thử — **quá chậm**. Nhưng ta đang "giải phương trình" `arr[i] ^ v = e` bằng cách đoán mò, trong khi XOR có **phép nghịch đảo** ngay trong tay!

## 4. Insight — XOR hai vế với `arr[i]`

Từ `encoded[i] = arr[i] ^ arr[i+1]`, XOR hai vế với `arr[i]`:

`arr[i] ^ encoded[i] = arr[i] ^ arr[i] ^ arr[i+1] = 0 ^ arr[i+1] = arr[i+1]`.

⇒ **`arr[i+1] = arr[i] ^ encoded[i]`**. Biết `arr[0] = first`, ta tính lần lượt `arr[1], arr[2], ...`.

In [7]:
a0, e0 = 4, 6
a1 = a0 ^ e0
print("arr[1] = arr[0] ^ encoded[0] =", a0, "^", e0, "=", a1)
print("kiểm tra: arr[0] ^ arr[1] =", a0 ^ a1, "== encoded[0] =", e0)

arr[1] = arr[0] ^ encoded[0] = 4 ^ 6 = 2
kiểm tra: arr[0] ^ arr[1] = 6 == encoded[0] = 6


## 5. Lời giải chính — giải mã tuần tự

**Đoạn 1:** `arr = [first]`.
**Đoạn 2:** với mỗi `e` trong `encoded`: `arr.append(arr[-1] ^ e)` (`arr[-1]` là phần tử vừa giải mã).
**Đoạn 3:** trả `arr`.

In [8]:
def decode_sequential(encoded: List[int], first: int) -> List[int]:
    arr = [first]
    for e in encoded:
        arr.append(arr[-1] ^ e)
    return arr

print(decode_sequential([1, 2, 3], 1))       # [1, 0, 2, 1]
print(decode_sequential([6, 2, 7, 3], 4))    # [4, 2, 0, 7, 4]

[1, 0, 2, 1]
[4, 2, 0, 7, 4]


## 6. TRACE từng bước

Ví dụ `encoded = [6, 2, 7, 3]`, `first = 4`.

| i | arr[i] (bit) | encoded[i] (bit) | arr[i+1] = arr[i] ^ encoded[i] |
|---:|---|---|---|
| 0 | 4 (`100`) | 6 (`110`) | `010` = 2 |
| 1 | 2 (`010`) | 2 (`010`) | `000` = 0 |
| 2 | 0 (`000`) | 7 (`111`) | `111` = 7 |
| 3 | 7 (`111`) | 3 (`011`) | `100` = 4 |

Kết quả `[4, 2, 0, 7, 4]`.

In [9]:
def trace_decode(encoded: List[int], first: int) -> List[int]:
    arr = [first]
    width = max([first] + encoded).bit_length() or 1     # số bit cần để in
    print(f"arr[0] = first = {first} ({format(first, f'0{width}b')})")
    for i, e in enumerate(encoded):
        nxt = arr[-1] ^ e
        print(f"i={i}: {format(arr[-1], f'0{width}b')} ^ {format(e, f'0{width}b')} = "
              f"{format(nxt, f'0{width}b')} -> arr[{i + 1}] = {nxt}")
        arr.append(nxt)
    print("arr =", arr, "| mã hoá lại:", encode(arr), "== encoded?", encode(arr) == encoded)
    return arr

trace_decode([6, 2, 7, 3], 4)
print()
trace_decode([1, 2, 3], 1)

arr[0] = first = 4 (100)
i=0: 100 ^ 110 = 010 -> arr[1] = 2
i=1: 010 ^ 010 = 000 -> arr[2] = 0
i=2: 000 ^ 111 = 111 -> arr[3] = 7
i=3: 111 ^ 011 = 100 -> arr[4] = 4
arr = [4, 2, 0, 7, 4] | mã hoá lại: [6, 2, 7, 3] == encoded? True

arr[0] = first = 1 (01)
i=0: 01 ^ 01 = 00 -> arr[1] = 0
i=1: 00 ^ 10 = 10 -> arr[2] = 2
i=2: 10 ^ 11 = 01 -> arr[3] = 1
arr = [1, 0, 2, 1] | mã hoá lại: [1, 2, 3] == encoded? True


[1, 0, 2, 1]

`x.bit_length()` trả về số bit cần để biểu diễn `x` (ví dụ `7 → 3`, `0 → 0`), dùng để in nhị phân thẳng hàng.

## 7. Các lời giải khác

### 7.1. Cấp phát trước mảng kích thước `n`

Giống lời giải chính nhưng dùng chỉ số thay vì `append` — cách viết quen thuộc trong C/Java.

In [10]:
def decode_index(encoded: List[int], first: int) -> List[int]:
    n = len(encoded) + 1
    arr = [0] * n
    arr[0] = first
    for i in range(n - 1):
        arr[i + 1] = arr[i] ^ encoded[i]
    return arr

print(decode_index([6, 2, 7, 3], 4))

[4, 2, 0, 7, 4]


### 7.2. XOR tiền tố (prefix XOR) — công thức trực tiếp cho `arr[k]`

Khai triển: `arr[k] = first ^ encoded[0] ^ encoded[1] ^ ... ^ encoded[k-1]` (các phần tử ở giữa triệt tiêu nhau). Vậy `arr` chính là **XOR tiền tố** của dãy `[first] + encoded`.

`itertools.accumulate(seq, func)` trả về các kết quả **cộng dồn** theo hàm `func`; `operator.xor(a, b)` chính là `a ^ b`.

In [11]:
from itertools import accumulate
from operator import xor

print("accumulate cộng:", list(accumulate([1, 2, 3, 4])))
print("accumulate xor :", list(accumulate([4, 6, 2, 7, 3], xor)))

def decode_accumulate(encoded: List[int], first: int) -> List[int]:
    return list(accumulate([first] + encoded, xor))

print(decode_accumulate([1, 2, 3], 1), decode_accumulate([6, 2, 7, 3], 4))

accumulate cộng: [1, 3, 6, 10]
accumulate xor : [4, 2, 0, 7, 4]
[1, 0, 2, 1] [4, 2, 0, 7, 4]


Vì sao các phần tử giữa triệt tiêu? Ví dụ `k = 2`:
`first ^ e0 ^ e1 = a0 ^ (a0 ^ a1) ^ (a1 ^ a2) = (a0 ^ a0) ^ (a1 ^ a1) ^ a2 = a2`. Dùng giao hoán + kết hợp để gom cặp, rồi `x ^ x = 0`.

In [12]:
a = [4, 2, 0, 7, 4]
e = encode(a)
k = 3
val = a[0]
for j in range(k):
    val ^= e[j]
print(f"first ^ e[0] ^ ... ^ e[{k - 1}] = {val} == arr[{k}] = {a[k]}")

first ^ e[0] ^ ... ^ e[2] = 7 == arr[3] = 7


### 7.3. Bảng so sánh

`n = len(arr)`, `V = 2^17` (giới hạn giá trị).

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| Dò thử | thử mọi `v < V` | `O(n·V)` | `O(1)` | quá chậm |
| **Tuần tự `append`** | `arr[i+1] = arr[i] ^ e[i]` | `O(n)` | `O(1)` (ngoài output) | khuyên dùng |
| Cấp phát trước | như trên, dùng chỉ số | `O(n)` | `O(1)` | phong cách C/Java |
| `accumulate(xor)` | XOR tiền tố | `O(n)` | `O(n)` (list tạm `[first] + encoded`) | gọn, "functional" |

## 8. Code nộp LeetCode

In [13]:
class Solution:
    def decode(self, encoded: List[int], first: int) -> List[int]:
        arr = [first]
        for e in encoded:
            arr.append(arr[-1] ^ e)
        return arr

sol = Solution()
print(sol.decode([1, 2, 3], 1), sol.decode([6, 2, 7, 3], 4))

[1, 0, 2, 1] [4, 2, 0, 7, 4]


Bản một dòng:

In [14]:
class SolutionAccumulate:
    def decode(self, encoded: List[int], first: int) -> List[int]:
        return list(accumulate([first] + encoded, xor))

print(SolutionAccumulate().decode([6, 2, 7, 3], 4))

[4, 2, 0, 7, 4]


## 9. Vì sao đúng và vì sao duy nhất?

**Đúng (quy nạp):**
- `arr[0] = first` đúng theo đề.
- Giả sử `arr[i]` đúng. Từ `encoded[i] = arr[i] ^ arr[i+1]` và tính chất `x ^ x = 0`, `x ^ 0 = x`: `arr[i] ^ encoded[i] = arr[i+1]`. Vậy giá trị ta tính cho `arr[i+1]` đúng.

**Duy nhất:** với `arr[i]` cố định, hàm `v ↦ arr[i] ^ v` là **song ánh** (XOR với cùng một số hai lần thì trở lại) ⇒ phương trình `arr[i] ^ v = encoded[i]` có **đúng một** nghiệm `v`. Lặp lại cho mọi `i` ⇒ `arr` duy nhất.

**Invariant của vòng lặp:** sau khi xử lý `encoded[0..i-1]`, list `arr` chứa đúng `i + 1` phần tử đầu của mảng gốc, và `arr[-1]` là phần tử thứ `i`.

## 10. Độ phức tạp

`n = len(encoded) + 1`.
- **Time `O(n)`:** mỗi phần tử một phép XOR và một `append` (trung bình `O(1)` — amortized).
- **Space:** `O(n)` cho mảng kết quả (bắt buộc); bộ nhớ phụ `O(1)`.
- Cách dò thử: `O(n · 2^17)`.

In [15]:
import random, time

random.seed(1720)
arr_big = [random.randint(0, 10**5) for _ in range(10**4)]
enc_big = encode(arr_big)
t = time.perf_counter(); r1 = decode_sequential(enc_big, arr_big[0]); t1 = time.perf_counter() - t
small = arr_big[:30]
t = time.perf_counter(); r2 = decode_search(encode(small), small[0]); t2 = time.perf_counter() - t
print(f"tuần tự n = 10^4: {t1:.4f}s, đúng = {r1 == arr_big}")
print(f"dò thử chỉ n = 30: {t2:.4f}s, đúng = {r2 == small}  (n = 10^4 sẽ chậm hơn ~300 lần)")

tuần tự n = 10^4: 0.0005s, đúng = True
dò thử chỉ n = 30: 0.0571s, đúng = True  (n = 10^4 sẽ chậm hơn ~300 lần)


## 11. Bộ test

1. Hai ví dụ đề.
2. Biên: `n = 2` (encoded 1 phần tử); `first = 0`; `encoded` toàn `0` (mảng hằng).
3. Dễ sai: giá trị lớn gần `10^5`; phần tử lặp lại.
4. **Kiểm thử khứ hồi (round-trip):** sinh `arr` ngẫu nhiên → `encode` → `decode` phải ra đúng `arr`; đối chiếu mọi cách.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

S = Solution()
check("đề 1", S.decode([1, 2, 3], 1), [1, 0, 2, 1])
check("đề 2", S.decode([6, 2, 7, 3], 4), [4, 2, 0, 7, 4])
check("n = 2", S.decode([5], 3), [3, 6])
check("first = 0", S.decode([7, 7], 0), [0, 7, 0])
check("encoded toàn 0", S.decode([0, 0, 0], 9), [9, 9, 9, 9])
check("giá trị lớn", S.decode([100000 ^ 99999], 100000), [100000, 99999])

PASS đề 1: [1, 0, 2, 1]
PASS đề 2: [4, 2, 0, 7, 4]
PASS n = 2: [3, 6]
PASS first = 0: [0, 7, 0]
PASS encoded toàn 0: [9, 9, 9, 9]
PASS giá trị lớn: [100000, 99999]


In [17]:
impls = [decode_sequential, decode_index, decode_accumulate, Solution().decode, SolutionAccumulate().decode]
random.seed(99)
for _ in range(2000):
    n = random.randint(2, 30)
    arr = [random.randint(0, 10**5) for _ in range(n)]
    enc = encode(arr)
    for f in impls:
        assert f(enc, arr[0]) == arr, (f, arr)
for _ in range(20):                                  # cách dò thử: chỉ vài bộ nhỏ
    arr = [random.randint(0, 50) for _ in range(5)]
    assert decode_search(encode(arr), arr[0], limit=64) == arr
print("Round-trip encode -> decode trên 2000 mảng ngẫu nhiên, 5 cài đặt: PASS")

Round-trip encode -> decode trên 2000 mảng ngẫu nhiên, 5 cài đặt: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — XOR với `encoded[i]` của **chính chỉ số sai**

Viết `arr[i] = arr[i-1] ^ encoded[i]` (lệch chỉ số) → sai và có thể `IndexError` ở cuối.

In [18]:
def wrong_index(encoded, first):
    n = len(encoded) + 1
    arr = [first] + [0] * (n - 1)
    for i in range(1, n):
        arr[i] = arr[i - 1] ^ encoded[i]     # SAI: phải là encoded[i - 1]
    return arr

try:
    print(wrong_index([6, 2, 7, 3], 4))
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Sửa:** `arr[i] = arr[i - 1] ^ encoded[i - 1]`, hoặc dùng cách `append` để khỏi lo chỉ số.

### Lỗi 2 — XOR với `encoded` gốc thay vì với phần tử **vừa giải mã**

Viết `arr.append(first ^ e)` cho mọi `e` → chỉ đúng phần tử thứ hai.

In [19]:
def wrong_use_first(encoded, first):
    return [first] + [first ^ e for e in encoded]      # SAI

print("sai:", wrong_use_first([6, 2, 7, 3], 4), "| đúng:", decode_sequential([6, 2, 7, 3], 4))

sai: [4, 2, 6, 3, 7] | đúng: [4, 2, 0, 7, 4]


**Sửa:** XOR với `arr[-1]` (phần tử liền trước), hoặc với **XOR tiền tố** của `encoded`.

### Lỗi 3 — Nhầm `^` với luỹ thừa

Từ toán học, nhiều bạn viết `2^3` để chỉ 8. Trong Python `2 ^ 3 = 1` (XOR), luỹ thừa là `2 ** 3`.

In [20]:
print("2 ^ 3 =", 2 ^ 3, "| 2 ** 3 =", 2 ** 3, "| 1 << 3 =", 1 << 3)

2 ^ 3 = 1 | 2 ** 3 = 8 | 1 << 3 = 8


### Lỗi 4 — Thử "trừ" thay vì XOR

Nghĩ "encoded là tổng/hiệu" rồi viết `arr[i+1] = encoded[i] - arr[i]`. Chỉ tình cờ đúng khi các bit không chồng nhau.

In [21]:
def wrong_subtract(encoded, first):
    arr = [first]
    for e in encoded:
        arr.append(e - arr[-1])        # SAI
    return arr

print("sai:", wrong_subtract([6, 2, 7, 3], 4), "| đúng:", decode_sequential([6, 2, 7, 3], 4))

sai: [4, 2, 0, 7, -4] | đúng: [4, 2, 0, 7, 4]


**Sửa:** phép ngược của XOR là **chính XOR**.

## 13. Mẹo nhận dạng pattern

- Thấy `a XOR b` đã biết và một trong hai số đã biết → số còn lại = **XOR lại** (`x ^ y ^ y = x`).
- XOR các phần tử liên tiếp → nghĩ tới **XOR tiền tố**: `pre[j] ^ pre[i] = XOR đoạn (i, j]`, giống tổng tiền tố.
- "Mọi số xuất hiện hai lần trừ một số" → XOR tất cả (cặp triệt tiêu).
- Bài liên quan: 136 Single Number, 268 Missing Number, 1310 XOR Queries of a Subarray, 1734 Decode XORed Permutation (khó hơn: không cho `first`), 1486 XOR Operation in an Array.

## 14. Biến thể

### Biến thể 1 — Cho **phần tử cuối** thay vì phần tử đầu

Giải mã ngược từ phải sang trái: `arr[i] = arr[i+1] ^ encoded[i]`.

In [22]:
def decode_from_last(encoded, last):
    arr = [last]
    for e in reversed(encoded):
        arr.append(arr[-1] ^ e)
    return arr[::-1]

print(decode_from_last([6, 2, 7, 3], 4), decode_from_last([1, 2, 3], 1))

[4, 2, 0, 7, 4] [1, 0, 2, 1]


### Biến thể 2 — XOR Queries (LeetCode 1310): XOR của đoạn `arr[l..r]` nhanh `O(1)`

`pre[0] = 0`, `pre[i+1] = pre[i] ^ arr[i]`. XOR đoạn `[l, r]` = `pre[r+1] ^ pre[l]`.

In [23]:
def xor_queries(arr, queries):
    pre = [0]
    for x in arr:
        pre.append(pre[-1] ^ x)
    return [pre[r + 1] ^ pre[l] for l, r in queries]

print(xor_queries([1, 3, 4, 8], [[0, 1], [1, 2], [0, 3], [3, 3]]))   # [2, 7, 14, 8]

[2, 7, 14, 8]


### Biến thể 3 — Single Number (LeetCode 136)

Mọi số xuất hiện **hai lần** trừ một số. XOR tất cả: cặp triệt tiêu, còn lại số lẻ loi.

In [24]:
def single_number(nums):
    r = 0
    for x in nums:
        r ^= x
    return r

print(single_number([4, 1, 2, 1, 2]), single_number([7]))

4 7


## 15. Bài tập (tăng dần)

> Hàm bài tập chỉ có `pass`, **không được gọi** khi chạy notebook.

### Bài 1 — Tính XOR bằng tay
Tính `5 ^ 3`, `12 ^ 10`, `9 ^ 9 ^ 4` bằng bảng bit (không chạy code).

In [25]:
def exercise_1():
    # TODO: trả về tuple 3 kết quả
    pass

# assert exercise_1() == (5 ^ 3, 12 ^ 10, 9 ^ 9 ^ 4)

### Bài 2 — Viết hàm `encode` không dùng list comprehension

In [26]:
def encode_loop(arr: List[int]) -> List[int]:
    # TODO: vòng for i in range(len(arr) - 1)
    pass

# assert encode_loop([1, 0, 2, 1]) == [1, 2, 3]

### Bài 3 — Điền chỗ trống lời giải

In [27]:
def decode_fill(encoded: List[int], first: int) -> List[int]:
    arr = [first]
    # TODO: với mỗi e, thêm arr[-1] ^ e
    pass

# assert decode_fill([6, 2, 7, 3], 4) == [4, 2, 0, 7, 4]

### Bài 4 — Chỉ lấy phần tử thứ `k` trong `O(k)` không tạo cả mảng

In [28]:
def kth_element(encoded: List[int], first: int, k: int) -> int:
    # TODO: XOR tiền tố tới encoded[k-1]
    pass

# assert kth_element([6, 2, 7, 3], 4, 3) == 7 and kth_element([6, 2, 7, 3], 4, 0) == 4

### Bài 5 — Biến thể: mã hoá bằng "cách một"
`encoded[i] = arr[i] ^ arr[i + 2]` (dài `n - 2`), cho **hai** phần tử đầu `a0, a1`. Giải mã.

In [29]:
def decode_skip(encoded: List[int], a0: int, a1: int) -> List[int]:
    # TODO
    pass

# arr = [3, 9, 4, 1, 7, 7]
# assert decode_skip([arr[i] ^ arr[i + 2] for i in range(len(arr) - 2)], 3, 9) == arr

### Bài 6 — Mini challenge: Decode XORed Permutation (LeetCode 1734)
`perm` là hoán vị của `1..n` (`n` **lẻ**), `encoded[i] = perm[i] ^ perm[i+1]`, **không** cho `first`. Tìm `perm`.

In [30]:
def decode_permutation(encoded: List[int]) -> List[int]:
    # TODO: XOR(1..n) = XOR mọi perm; XOR encoded[1], encoded[3], ... = XOR perm[1..n-1]
    pass

# assert decode_permutation([3, 1]) == [1, 2, 3]
# assert decode_permutation([6, 5, 4, 6]) == [2, 4, 1, 5, 3]

## 16. Gợi ý 3 mức

**Bài 1:** (1) Viết 4 bit. (2) `0101 ^ 0011`, `1100 ^ 1010`. (3) `6, 6, 4` (`9 ^ 9 = 0`).

**Bài 2:** (1) Tạo list rỗng. (2) `res.append(arr[i] ^ arr[i + 1])`. (3) `return res`.

**Bài 3:** (1) Như mục 5. (2) `for e in encoded: arr.append(arr[-1] ^ e)`. (3) `return arr`.

**Bài 4:** (1) `arr[k] = first ^ e[0] ^ ... ^ e[k-1]`. (2) Vòng `for j in range(k)`. (3) `val ^= encoded[j]`.

**Bài 5:** (1) `arr[i+2] = arr[i] ^ encoded[i]`. (2) Bắt đầu `arr = [a0, a1]`. (3) `arr.append(arr[i] ^ encoded[i])` với `i` tăng dần.

**Bài 6:** (1) `total = 1 ^ 2 ^ ... ^ n`. (2) `encoded[1] ^ encoded[3] ^ ...` = `perm[1] ^ perm[2] ^ ... ^ perm[n-1]`. (3) `first = total ^ (đại lượng đó)`, rồi giải mã như bài gốc.

## 17. Đáp án tham khảo

In [31]:
def ans_exercise_1():
    return (6, 6, 4)

def ans_encode_loop(arr):
    res = []
    for i in range(len(arr) - 1):
        res.append(arr[i] ^ arr[i + 1])
    return res

def ans_decode_fill(encoded, first):
    arr = [first]
    for e in encoded:
        arr.append(arr[-1] ^ e)
    return arr

def ans_kth_element(encoded, first, k):
    val = first
    for j in range(k):
        val ^= encoded[j]
    return val

def ans_decode_skip(encoded, a0, a1):
    arr = [a0, a1]
    for i, e in enumerate(encoded):
        arr.append(arr[i] ^ e)
    return arr

def ans_decode_permutation(encoded):
    n = len(encoded) + 1
    total = 0
    for v in range(1, n + 1):
        total ^= v
    rest = 0
    for i in range(1, len(encoded), 2):     # encoded[1], encoded[3], ...
        rest ^= encoded[i]
    first = total ^ rest
    return ans_decode_fill(encoded, first)

assert ans_exercise_1() == (5 ^ 3, 12 ^ 10, 9 ^ 9 ^ 4)
assert ans_encode_loop([1, 0, 2, 1]) == [1, 2, 3]
assert ans_decode_fill([6, 2, 7, 3], 4) == [4, 2, 0, 7, 4]
assert ans_kth_element([6, 2, 7, 3], 4, 3) == 7 and ans_kth_element([6, 2, 7, 3], 4, 0) == 4
arr_s = [3, 9, 4, 1, 7, 7]
assert ans_decode_skip([arr_s[i] ^ arr_s[i + 2] for i in range(len(arr_s) - 2)], 3, 9) == arr_s
assert ans_decode_permutation([3, 1]) == [1, 2, 3]
assert ans_decode_permutation([6, 5, 4, 6]) == [2, 4, 1, 5, 3]
for _ in range(300):
    n = random.choice([3, 5, 7, 9, 11])
    perm = random.sample(range(1, n + 1), n)
    assert ans_decode_permutation(encode(perm)) == perm
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def decode(self, encoded: List[int], first: int) -> List[int]:
        arr = [first]
        for e in encoded:
            arr.append(arr[-1] ^ e)     # a ^ (a ^ b) = b
        return arr
```

**Khung tư duy:** biết `a ^ b` và biết `a` ⇒ `b = a ^ (a ^ b)`; chuỗi liên tiếp ⇒ giải tuần tự hoặc XOR tiền tố.

**Checklist trước khi submit**
- [ ] Dùng `^` (XOR), không phải `**` hay phép trừ?
- [ ] XOR với phần tử **vừa giải mã** (`arr[-1]`), không phải `first`?
- [ ] Kết quả dài `len(encoded) + 1`?
- [ ] Đã thử round-trip `encode(decode(...))`?
- [ ] Tên method `decode`, trả về `List[int]`?